# Coding LLM server on Kaggle (GPU T4 x2): llama.cpp + Cloudflare quick tunnel

Started by `novel kllm up` (it pushes this notebook with a small private dataset, `kllm-session`, that holds the API key,
the limits and a copy of `novel/kllm.py`). The notebook installs a pinned llama.cpp, downloads the GGUF model from
Hugging Face, starts llama-server on 127.0.0.1, puts the streaming proxy from `novel/kllm.py` in front of it, opens a
Cloudflare quick tunnel and posts the tunnel address (never the key) to ntfy.sh under a random topic. It ends itself after
the idle time without an inference request, on `novel kllm down`, at the max-hours cap, or if llama-server dies.
Its summary (timings, context reached, VRAM) is left in `/kaggle/working/kllm_summary.json`.

`smoke` sessions run the same code on a CPU kernel with a tiny model, to test the plumbing without GPU quota.


In [ ]:
# Cell 1: the session, the mode, the watchdog. Cheap checks only; nothing is installed or downloaded here.
import atexit, json, os, re, shutil, subprocess, sys, threading, time
import urllib.error, urllib.request
from pathlib import Path

T0 = time.time()


def _find_dataset(slug: str) -> Path:
    for c in [Path('/kaggle/input') / slug, *Path('/kaggle/input/datasets').glob(f'*/{slug}')]:
        if c.exists():
            return c
    raise FileNotFoundError(f"dataset '{slug}' not found under /kaggle/input: attach it to the kernel")


def _secret(slug: str, name: str) -> str:
    p = _find_dataset(slug) / name
    if not p.exists():
        raise FileNotFoundError(f"dataset '{slug}' has no file {name}")
    v = p.read_text().strip()
    assert v, f"{slug}/{name} is empty"
    return v


SRC = _find_dataset('kllm-session')
session = json.loads((SRC / 'session.json').read_text(encoding='utf-8'))
sys.path.insert(0, str(SRC))
import kllm

KEY = session["key"]                       # never printed, never posted anywhere but the private session dataset
assert len(KEY) >= 16 and session.get("topic"), "session.json has no key or topic"
TOPIC = session["topic"]
SMOKE = bool(session.get("smoke"))         # CPU kernel, tiny model: proves the plumbing without GPU quota
GPU = not SMOKE
IDLE_MIN = float(session["idle_minutes"])
MAX_H = float(session["max_hours"])
THINKING = bool(session.get("thinking"))
ALIAS = session.get("alias", "qwen-local")
PAD = int(session.get("pad_bytes", kllm.DEFAULT_PAD))
CONTEXTS = [4096] if SMOKE else [int(c) for c in session["contexts"]]
HOLD_S = 150 * 60 if GPU else 20 * 60      # hard ceiling on the setup phase: the kernel must not sit half-started
HARD_S = int(MAX_H * 3600) + 600           # the watchdog: whatever hangs, the kernel is gone by then
LLAMA_TAG = "b12430"                       # pinned: the flags below were measured at this tag
PORT, FRONT_PORT = 8080, 8081
CLOUDFLARED = "https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64"
if GPU:
    MODEL = {"repo": "unsloth/Qwen3.6-35B-A3B-GGUF", "file": "Qwen3.6-35B-A3B-UD-Q4_K_M.gguf", "size": 22134528992}
else:
    MODEL = {"repo": "Qwen/Qwen2.5-0.5B-Instruct-GGUF", "file": "qwen2.5-0.5b-instruct-q4_k_m.gguf", "size": 491400032}

OUT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("kllm_out")
OUT.mkdir(parents=True, exist_ok=True)
SUMMARY = {"mode": "smoke" if SMOKE else "gpu", "llama_tag": LLAMA_TAG, "contexts_tried": [], "timings_s": {}}
CHILDREN = []


def log(*a):
    print(f"[{time.time() - T0:6.0f}s]", *a, flush=True)


def save():
    (OUT / "kllm_summary.json").write_text(json.dumps(SUMMARY, indent=1, default=str), encoding="utf-8")


def _kill_children():
    for p in CHILDREN:
        try:
            if p.poll() is None:
                p.kill()
        except Exception:
            pass


atexit.register(_kill_children)


def _watchdog():
    while time.time() - T0 < HARD_S:
        time.sleep(2)
    print(f"\nKLLM WATCHDOG: hard limit of {HARD_S}s reached; killing children and exiting", flush=True)
    SUMMARY["watchdog"] = f"hard limit {HARD_S}s"
    try:
        save()
    except Exception:
        pass
    _kill_children()
    sys.stdout.flush()
    os._exit(3)


threading.Thread(target=_watchdog, daemon=True).start()


def fatal(msg: str):
    SUMMARY["fatal"] = msg
    save()
    print("KLLM FATAL:", msg, flush=True)
    raise RuntimeError(msg)


def sh(cmd, timeout=120, env=None):
    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, timeout=timeout, env=env)
    return p.returncode, ((p.stdout or "") + (p.stderr or "")).strip()


def setup_left() -> float:
    return HOLD_S - (time.time() - T0)


print("KLLM session:", {k: v for k, v in session.items() if k not in ("key", "topic")})

# ---- GPU inventory
if GPU:
    rc, full = sh(["nvidia-smi"])
    print(full)
    rc, q = sh(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader,nounits"])
    gpus = [x.strip() for x in q.splitlines() if x.strip()] if rc == 0 else []
    SUMMARY["gpus"] = gpus
    if len(gpus) != 2 or not all("T4" in g for g in gpus):
        fatal(f"expected 2 x T4, nvidia-smi says {gpus!r} (kernel pushed without machine_shape NvidiaTeslaT4?)")
    DRIVER_MAJOR = int(gpus[0].split(",")[3].strip().split(".")[0])
else:
    DRIVER_MAJOR = 0

# ---- scratch directory: not /kaggle/working (output cap, and everything in it is kept as output)
need = MODEL["size"] + 4_000_000_000
WORK = None
for cand in ("/kaggle/temp", "/tmp/kllm"):
    p = Path(cand)
    try:
        p.mkdir(parents=True, exist_ok=True)
        free = shutil.disk_usage(p).free
    except OSError:
        continue
    print(f"scratch candidate {p}: {free / 1e9:.1f} GB free")
    if free >= need:
        WORK = p
        break
if WORK is None:
    fatal(f"no scratch directory with {need / 1e9:.0f} GB free")
log("KLLM mode", SUMMARY["mode"], "scratch", WORK, "idle min", IDLE_MIN, "max h", MAX_H, "contexts", CONTEXTS)
save()

In [ ]:
# Cell 2: a llama-server binary. GPU: the official Ubuntu CUDA 12.8 release (+ the cudart bundle if libraries are missing).
# CPU smoke: the official CPU release. Both from the pinned tag; no source-build fallback (this must fail fast).
BASE = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
CPU_TGZ = f"llama-{LLAMA_TAG}-bin-ubuntu-x64.tar.gz"
CUDA_TGZ = f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
CUDART_TGZ = f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"


def fetch(name: str) -> Path:
    dest, t = WORK / name, time.time()
    rc, out = sh(["curl", "-fL", "--retry", "3", "--retry-delay", "3", "--connect-timeout", "20", "-sS",
                  "-o", str(dest), f"{BASE}/{name}"], timeout=900)
    if rc:
        fatal(f"download {name} failed: {out[-300:]}")
    log(f"downloaded {name} {dest.stat().st_size / 1e6:.0f} MB in {time.time() - t:.0f}s")
    return dest


def untar(tgz: Path, dest: Path):
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    rc, out = sh(["tar", "-xzf", str(tgz), "-C", str(dest)], timeout=300)
    if rc:
        fatal(f"extract {tgz.name}: {out[-300:]}")
    tgz.unlink()


def lib_env(bindir: Path) -> dict:
    return dict(os.environ, LD_LIBRARY_PATH=f"{bindir}:{os.environ.get('LD_LIBRARY_PATH', '')}")


def inspect_build(server: Path) -> dict:
    """Missing shared libraries (libcuda.so is the driver's) and whether `llama-server --version` runs."""
    bindir, env, miss = server.parent, lib_env(server.parent), {}
    for f in [server, *bindir.glob("libggml-cuda*.so*")]:
        p = subprocess.run(["ldd", str(f)], capture_output=True, text=True, env=env)
        for line in (p.stdout + p.stderr).splitlines():
            if "not found" in line and "=>" in line and not line.split("=>")[0].strip().startswith("libcuda.so"):
                miss.setdefault(f.name, []).append(line.split("=>")[0].strip())
    rc, out = sh([str(server), "--version"], timeout=60, env=env)
    return {"missing_libs": miss, "version_rc": rc, "version_out": out[-300:]}


t_install = time.time()
untar(fetch(CUDA_TGZ if GPU else CPU_TGZ), WORK / "llama")
exes = [p for p in (WORK / "llama").rglob("llama-server") if p.is_file()]
if not exes:
    fatal("no llama-server in the release archive")
SERVER = exes[0]
info = inspect_build(SERVER)
if GPU and info["missing_libs"]:
    log("missing libs, adding the cudart bundle:", info["missing_libs"])
    untar(fetch(CUDART_TGZ), WORK / "cudart")
    for f in (WORK / "cudart").rglob("*.so*"):
        if f.is_file():
            shutil.copy2(f, SERVER.parent / f.name)
    shutil.rmtree(WORK / "cudart", ignore_errors=True)
    info = inspect_build(SERVER)
if info["version_rc"] != 0 or info["missing_libs"]:
    fatal(f"llama-server does not run here: {info}" + (f" (driver {DRIVER_MAJOR})" if GPU else ""))
if GPU and DRIVER_MAJOR < 570:
    fatal(f"driver {DRIVER_MAJOR} is older than 570: the CUDA 12.8 release cannot run")
rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub[hf_xet]"], timeout=300)
if rc:
    fatal("pip install huggingface_hub failed: " + out[-300:])
SUMMARY["timings_s"]["install"] = round(time.time() - t_install, 1)
log("KLLM llama-server ready in", SUMMARY["timings_s"]["install"], "s:", info["version_out"].splitlines()[-1:])
save()

In [ ]:
# Cell 3: the model (from Hugging Face, every start) and llama-server, trying the context sizes in order.
DL = ("import sys; from huggingface_hub import hf_hub_download; "
      "p = hf_hub_download(sys.argv[1], sys.argv[2], local_dir=sys.argv[3]); print('PATH', p)")
try:
    HF_TOKEN = _secret('hf-token', 'HF_TOKEN')        # avoids the anonymous rate limit that failed a benchmark once
except Exception as e:
    HF_TOKEN = None
    log("no HF token:", e)


def tree_bytes(d: Path) -> int:
    return sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) if d.exists() else 0


def download_once(dest: Path, env: dict, deadline: float):
    p = subprocess.Popen([sys.executable, "-c", DL, MODEL["repo"], MODEL["file"], str(dest)], env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    CHILDREN.append(p)
    last_t, last_n = time.time(), tree_bytes(dest)
    last_growth = time.time()
    while p.poll() is None:
        time.sleep(15 if GPU else 3)
        n, now = tree_bytes(dest), time.time()
        if n > last_n:
            last_growth = now
        log(f"download: {n / 1e9:.2f} of {MODEL['size'] / 1e9:.2f} GB, {(n - last_n) / max(now - last_t, 1) / 1e6:.0f} MB/s")
        last_t, last_n = now, n
        if now - last_growth > 240 or now > deadline:
            p.kill()
            return False, f"download stalled or too slow ({n / 1e9:.2f} GB)"
    out = p.stdout.read()
    path = dest / MODEL["file"]
    if p.returncode != 0 or not path.exists():
        return False, f"rc {p.returncode}: {out[-300:]}"
    if path.stat().st_size != MODEL["size"]:
        return False, f"{path.name}: {path.stat().st_size} bytes, expected {MODEL['size']}"
    return True, str(path)


def download_model() -> Path:
    """Up to 3 attempts: token with the default transfer, token over plain HTTP, then no token at all (a bad token
    makes Hugging Face answer 401 even for a public file)."""
    dest = WORK / "model"
    dest.mkdir(parents=True, exist_ok=True)
    deadline = time.time() + (25 * 60 if GPU else 3 * 60)
    errors = []
    for attempt, (pause, plain, token) in enumerate([(0, False, True), (30, True, True), (60, True, False)], 1):
        if attempt > 1:
            log(f"download attempt {attempt - 1} failed ({errors[-1][:160]}); waiting {pause}s")
            time.sleep(pause)
        if time.time() > deadline:
            break
        env = dict(os.environ, HF_HUB_DISABLE_PROGRESS_BARS="1", HF_HUB_DISABLE_TELEMETRY="1")
        if plain:
            env["HF_HUB_DISABLE_XET"] = "1"
        for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(k, None)
        if token and HF_TOKEN:
            env["HF_TOKEN"] = HF_TOKEN
        ok, msg = download_once(dest, env, deadline)
        if ok:
            return Path(msg)
        errors.append(msg)
    fatal("model download failed after retries: " + " | ".join(e[-250:] for e in errors))


def vram_used() -> list:
    if not GPU:
        return []
    rc, out = sh(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], timeout=30)
    return [int(x) for x in out.split() if x.strip().isdigit()] if rc == 0 else []


def local(method, path, body=None, timeout=300):
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}{path}", method=method,
                                 data=None if body is None else json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, r.read()


class Server:
    def __init__(self, model: Path, ctx: int):
        self.ctx, self.logfile, self.proc = ctx, WORK / f"server_{ctx}.log", None
        self.args = [str(SERVER), "-m", str(model), "-c", str(ctx), "-ngl", "all" if GPU else "0", "-sm", "layer",
                     "-fa", "on", "-ctk", "q8_0", "-ctv", "q8_0", "-np", "1", "--jinja", "--host", "127.0.0.1",
                     "--port", str(PORT), "--no-webui", "-fit", "off", "--alias", ALIAS]
        if not THINKING:
            self.args += ["--chat-template-kwargs", '{"enable_thinking":false}']

    def log_text(self) -> str:
        try:
            return self.logfile.read_text(errors="replace")
        except OSError:
            return ""

    def tail(self, n=25) -> str:
        return "\n".join(self.log_text().splitlines()[-n:])

    def start(self, timeout: float) -> float:
        env = lib_env(SERVER.parent)
        env["CUDA_CACHE_PATH"] = str(WORK / "nvcache")
        env["CUDA_CACHE_MAXSIZE"] = "4294967296"
        self.f = open(self.logfile, "wb")
        self.proc = subprocess.Popen(self.args, stdout=self.f, stderr=subprocess.STDOUT, env=env)
        CHILDREN.append(self.proc)
        t0 = time.time()
        while True:
            if self.proc.poll() is not None:
                raise RuntimeError(f"llama-server exited with {self.proc.returncode} while loading:\n{self.tail()}")
            try:
                if local("GET", "/health", timeout=5)[0] == 200:
                    return time.time() - t0
            except Exception:
                pass
            if time.time() - t0 > timeout:
                raise RuntimeError(f"llama-server not ready after {timeout:.0f}s:\n{self.tail()}")
            time.sleep(3)

    def alive(self) -> bool:
        return self.proc is not None and self.proc.poll() is None

    def stop(self):
        if self.proc is not None and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(60)
            except subprocess.TimeoutExpired:
                self.proc.kill()
                self.proc.wait(30)
        try:
            self.f.close()
        except Exception:
            pass
        lines = self.log_text().splitlines()
        keep = lines if len(lines) <= 600 else lines[:200] + ["... trimmed ..."] + lines[-400:]
        (OUT / "logs").mkdir(exist_ok=True)
        (OUT / "logs" / self.logfile.name).write_text("\n".join(keep), encoding="utf-8")


t_dl = time.time()
MODEL_PATH = download_model()
SUMMARY["timings_s"]["download"] = round(time.time() - t_dl, 1)
log("KLLM model downloaded in", SUMMARY["timings_s"]["download"], "s")

SRV = None
for ctx in CONTEXTS:
    if setup_left() < 120:
        fatal("setup ceiling reached before a server was up")
    srv = Server(MODEL_PATH, ctx)
    row = {"ctx": ctx}
    SUMMARY["contexts_tried"].append(row)
    try:
        row["load_s"] = round(srv.start(timeout=min(10 * 60 if GPU else 3 * 60, setup_left() - 60)), 1)
        st, raw = local("POST", "/v1/chat/completions", {"model": ALIAS, "max_tokens": 8, "messages": [
            {"role": "user", "content": "Say hello."}]})
        assert st == 200, f"self-test HTTP {st}"
        row["vram_mib"] = vram_used()
        row["ok"] = True
        SRV = srv
        break
    except Exception as e:
        row["ok"], row["error"] = False, str(e)[-700:]
        log(f"KLLM context {ctx} failed: {str(e)[-300:]}")
        srv.stop()
        for _ in range(30):                                  # let the cards empty before the next try
            v = vram_used()
            if not v or max(v) < 800:
                break
            time.sleep(3)
    save()
if SRV is None:
    fatal("llama-server could not start at any context size: " + json.dumps(SUMMARY["contexts_tried"])[:1500])
CTX = SRV.ctx
SUMMARY.update(ctx=CTX, vram_mib=vram_used())
SUMMARY["timings_s"]["model_ready"] = round(time.time() - T0, 1)
log(f"KLLM llama-server up: context {CTX}, VRAM {SUMMARY['vram_mib']} MiB per card, {SUMMARY['timings_s']['model_ready']}s since start")
save()

In [ ]:
# Cell 4: the proxy, the Cloudflare tunnel, the address relay, and the keep-alive loop that ends the kernel.
state = kllm.ProxyState(idle_minutes=IDLE_MIN, pad=PAD, info={"ctx": CTX, "model": ALIAS, "gguf": MODEL["file"],
                                                              "thinking": THINKING, "smoke": SMOKE})
PROXY = kllm.serve_llm_proxy(PORT, KEY, state, port=FRONT_PORT)
log(f"KLLM proxy on 127.0.0.1:{FRONT_PORT}, padding {PAD} bytes")

CF = WORK / "cloudflared"
urllib.request.urlretrieve(CLOUDFLARED, CF)
CF.chmod(0o755)


def start_tunnel():
    """cloudflared on the proxy port; returns (process, https address). Raises when no address appears in 2 minutes."""
    proc = subprocess.Popen([str(CF), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{FRONT_PORT}"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    CHILDREN.append(proc)
    found, box = threading.Event(), {}

    def drain():
        for line in proc.stdout:                 # keeps the pipe empty for the life of the tunnel
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m and not found.is_set():
                box["url"] = m.group(0)
                found.set()
    threading.Thread(target=drain, daemon=True).start()
    if not found.wait(120):
        proc.kill()
        raise RuntimeError("cloudflared printed no trycloudflare.com address in 2 minutes")
    return proc, box["url"]


def reachable(url: str, tries=45) -> bool:
    for _ in range(tries):
        try:
            if kllm.server_status(url, KEY, timeout=10).get("ok"):
                return True
        except kllm.KllmError:
            time.sleep(2)
    return False


def publish(url: str):
    """The PC reads the address from here (Kaggle shows no log while a notebook runs). The key never leaves the kernel."""
    msg = {"url": url, "ctx": CTX, "model": ALIAS, "ready_s": round(time.time() - T0, 1), "epoch": round(time.time())}
    print(f"KLLM URL {url}", flush=True)
    last = None
    for _ in range(18):
        try:
            kllm.relay_post(TOPIC, msg)
            log("KLLM address published to the relay")
            return
        except kllm.KllmError as e:
            last = e
            time.sleep(10)
    log(f"KLLM relay failed ({last}); use `novel kllm url {url}` on the PC")


TUNNEL = URL = None
for attempt in range(1, 4):
    if setup_left() < 60:
        fatal("setup ceiling reached before the tunnel was up")
    try:
        TUNNEL, URL = start_tunnel()
    except Exception as e:
        log(f"KLLM tunnel attempt {attempt} failed: {e}")
        continue
    if reachable(URL):
        break
    log(f"KLLM tunnel attempt {attempt}: {URL} does not answer from inside")
    TUNNEL.kill()
    TUNNEL = URL = None
if TUNNEL is None:
    fatal("no working Cloudflare tunnel after 3 attempts")
state.info["ready_s"] = SUMMARY["timings_s"]["ready"] = round(time.time() - T0, 1)
state.touch()                                    # the idle clock starts now
SUMMARY["url_host"] = URL.split("//")[1]
save()
publish(URL)
log(f"KLLM READY after {SUMMARY['timings_s']['ready']}s: context {CTX}, idle limit {IDLE_MIN:g} min, cap {MAX_H:g} h")

reason, restarts, tick = None, 0, 5 if SMOKE else 15
while reason is None:
    time.sleep(tick)
    if state.stop_event.is_set():
        reason = "stop requested"
    elif state.expired():
        reason = f"idle for {IDLE_MIN:g} minutes"
    elif time.time() - T0 > MAX_H * 3600:
        reason = f"{MAX_H:g} hour cap"
    elif not SRV.alive():
        reason = f"llama-server exited ({SRV.proc.returncode})"
    elif TUNNEL.poll() is not None:
        if restarts >= 1:
            reason = "the tunnel died twice"
        else:
            restarts += 1
            log("KLLM tunnel died; restarting it once")
            try:
                TUNNEL, URL = start_tunnel()
                if reachable(URL):
                    publish(URL)
                else:
                    reason = "the restarted tunnel does not answer"
            except Exception as e:
                reason = f"the tunnel could not be restarted: {e}"
    if int(time.time() - T0) % 300 < tick:
        log("KLLM hb", state.snapshot())

SUMMARY.update(stop_reason=reason, requests=state.requests, uptime_s=round(time.time() - T0))
log("KLLM STOP:", reason)
PROXY.shutdown()
if TUNNEL.poll() is None:
    TUNNEL.terminate()
SRV.stop()
save()
print("KLLM DONE", flush=True)